# HuiDi Model Training

This notebook is a reusable training workflow for HuiDi-style CSV datasets. Update the filename in the first code cell to train on a different CSV with the same schema.


In [21]:
# Quick environment check
%cd /home/jovyan/work
!ls

/home/jovyan/work
boosted_tree_submission.csv   HuiDis_Data_Exploration.ipynb  train.csv
decision_tree_submission.csv  random_forest_submission.csv   train_HuiDi_1.csv
FinalProject.ipynb	      README.md
HuiDi_Model_Training.ipynb    test.csv


In [22]:
# Configuration
TRAIN_CSV = "train_HuiDi_1.csv"
TARGET_COLUMN = "INDICATED_DAMAGE"
RANDOM_STATE = 42


In [23]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

pd.options.display.max_columns = 100


In [24]:
# Define feature groups and merge cleaning ideas from HuiDi exploration + FinalProject.
struck_map = {
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "10-Feb": 6,
    "2-10": 6,
    "11-100": 55,
    "Over 100": 150,
    "More than 100": 150,
}

base_columns = [
    "LATITUDE",
    "LONGITUDE",
    "AC_CLASS",
    "AC_MASS",
    "TYPE_ENG",
    "NUM_ENGS",
    "PHASE_OF_FLIGHT",
    "HEIGHT",
    "INCIDENT_MONTH",
    "TIME_OF_DAY",
    "SPECIES_ID",
    "SIZE",
    "NUM_STRUCK",
    "WARNED",
    TARGET_COLUMN,
]

engineered_columns = [
    "IS_MIGRATION_SEASON",
    "IS_LOW_VISIBILITY_TIME",
    "HEIGHT_MISSING",
    "MONTH_SIN",
    "MONTH_COS",
]

selected_columns = base_columns + engineered_columns

numeric_features = [
    "LATITUDE",
    "LONGITUDE",
    "AC_MASS",
    "NUM_ENGS",
    "HEIGHT",
    "INCIDENT_MONTH",
    "NUM_STRUCK",
    "IS_MIGRATION_SEASON",
    "IS_LOW_VISIBILITY_TIME",
    "HEIGHT_MISSING",
    "MONTH_SIN",
    "MONTH_COS",
]

categorical_features = [
    "AC_CLASS",
    "TYPE_ENG",
    "PHASE_OF_FLIGHT",
    "TIME_OF_DAY",
    "SPECIES_ID",
    "SIZE",
    "WARNED",
]

feature_columns = numeric_features + categorical_features


def _engineer_features(df):
    out = df.copy()

    # Convert NUM_STRUCK from grouped labels to numeric midpoints when needed.
    if "NUM_STRUCK" in out.columns:
        struck_as_string = out["NUM_STRUCK"].astype("string").str.strip()
        out["NUM_STRUCK"] = struck_as_string.map(struck_map).fillna(pd.to_numeric(out["NUM_STRUCK"], errors="coerce"))

    # Seasonal migration flag.
    if "INCIDENT_MONTH" in out.columns:
        month_numeric = pd.to_numeric(out["INCIDENT_MONTH"], errors="coerce")
        out["INCIDENT_MONTH"] = month_numeric
        out["IS_MIGRATION_SEASON"] = month_numeric.isin([3, 4, 5, 8, 9, 10, 11]).astype(int)
        out["MONTH_SIN"] = np.sin(2 * np.pi * (month_numeric - 1) / 12)
        out["MONTH_COS"] = np.cos(2 * np.pi * (month_numeric - 1) / 12)
    else:
        out["IS_MIGRATION_SEASON"] = np.nan
        out["MONTH_SIN"] = np.nan
        out["MONTH_COS"] = np.nan

    # Low visibility time flag.
    if "TIME_OF_DAY" in out.columns:
        time_text = out["TIME_OF_DAY"].astype("string").str.strip()
        out["TIME_OF_DAY"] = time_text.replace({pd.NA: np.nan}).astype(object)
        out["IS_LOW_VISIBILITY_TIME"] = time_text.isin(["Dawn", "Dusk", "Night"]).astype(int)
    else:
        out["IS_LOW_VISIBILITY_TIME"] = np.nan

    # Log-scale height and track missingness.
    if "HEIGHT" in out.columns:
        height_numeric = pd.to_numeric(out["HEIGHT"], errors="coerce")
        out["HEIGHT_MISSING"] = height_numeric.isna().astype(int)
        out["HEIGHT"] = np.log1p(height_numeric.clip(lower=0))
    else:
        out["HEIGHT_MISSING"] = np.nan

    return out


def clean_feature_frame(df, columns):
    cleaned = df.filter(items=columns).copy()

    for col in numeric_features:
        if col in cleaned.columns:
            cleaned[col] = pd.to_numeric(cleaned[col], errors="coerce")

    for col in categorical_features:
        if col in cleaned.columns:
            cleaned[col] = cleaned[col].replace(r"^\s*$", np.nan, regex=True)
            cleaned[col] = cleaned[col].where(pd.notna(cleaned[col]), np.nan).astype(object)

    return cleaned


def prepare_training_data(df):
    engineered = _engineer_features(df)
    cleaned = clean_feature_frame(engineered, selected_columns)
    cleaned = cleaned.dropna(subset=[TARGET_COLUMN]).copy()
    return cleaned[feature_columns + [TARGET_COLUMN]]


train_clean = prepare_training_data(train)
print("Cleaned dataset shape:", train_clean.shape)
display(train_clean.head())
print()
print("Feature dtypes:")
print(train_clean.dtypes)


Cleaned dataset shape: (307178, 20)


,LATITUDE,LONGITUDE,AC_MASS,NUM_ENGS,HEIGHT,INCIDENT_MONTH,NUM_STRUCK,IS_MIGRATION_SEASON,IS_LOW_VISIBILITY_TIME,HEIGHT_MISSING,MONTH_SIN,MONTH_COS,AC_CLASS,TYPE_ENG,PHASE_OF_FLIGHT,TIME_OF_DAY,SPECIES_ID,SIZE,WARNED,INDICATED_DAMAGE
0,18.439420,-66.001830,4.0,3.0,1.903168,12,6.0,0,0,0,-0.500000,8.660254e-01,A,D,Approach,Day,UNKBS,Small,No,0
1,2.745578,101.709917,4.0,3.0,1.595709,2,1.0,0,1,0,0.500000,8.660254e-01,A,D,Approach,Night,UNKBM,Medium,Unknown,0
2,38.174390,-85.736000,4.0,2.0,2.214934,5,1.0,1,1,0,0.866025,-5.000000e-01,A,D,Approach,Night,UNKBL,Large,No,1
3,33.942540,-118.408070,NaN,NaN,NaN,10,6.0,1,0,1,-1.000000,-1.836970e-16,NaN,NaN,NaN,NaN,NE120,Medium,Unknown,0
4,21.975980,-159.338960,4.0,2.0,0.000000,2,1.0,0,1,0,0.500000,8.660254e-01,A,D,Landing Roll,Dawn,R1101,Medium,No,0



Feature dtypes:
LATITUDE                  float64
LONGITUDE                 float64
AC_MASS                   float64
NUM_ENGS                  float64
HEIGHT                    float64
INCIDENT_MONTH              int64
NUM_STRUCK                float64
IS_MIGRATION_SEASON         int64
IS_LOW_VISIBILITY_TIME      int64
HEIGHT_MISSING              int64
MONTH_SIN                 float64
MONTH_COS                 float64
AC_CLASS                   object
TYPE_ENG                   object
PHASE_OF_FLIGHT            object
TIME_OF_DAY                object
SPECIES_ID                 object
SIZE                       object
WARNED                     object
INDICATED_DAMAGE            int64
dtype: object


In [25]:
# Split the dataset into train and validation sets.
X = train_clean.drop(columns=TARGET_COLUMN)
y = train_clean[TARGET_COLUMN]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

numeric_transformer = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median"))]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=10)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)


def evaluate_model(model_name, model):
    model.fit(X_train, y_train)
    predictions = model.predict(X_val)
    probabilities = model.predict_proba(X_val)[:, 1]

    print(f"{model_name} Results")
    print(f"Accuracy:           {accuracy_score(y_val, predictions):.4f}")
    print(f"Balanced Accuracy:  {balanced_accuracy_score(y_val, predictions):.4f}")
    print(f"Precision:          {precision_score(y_val, predictions):.4f}")
    print(f"Recall:             {recall_score(y_val, predictions):.4f}")
    print(f"F1 Score:           {f1_score(y_val, predictions):.4f}")
    print(f"ROC AUC:            {roc_auc_score(y_val, probabilities):.4f}")
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_val, predictions))
    print("\nClassification Report:")
    print(classification_report(y_val, predictions))
    print()

    return {
        "model": model_name,
        "accuracy": accuracy_score(y_val, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_val, predictions),
        "precision": precision_score(y_val, predictions),
        "recall": recall_score(y_val, predictions),
        "f1": f1_score(y_val, predictions),
        "roc_auc": roc_auc_score(y_val, probabilities),
    }


In [26]:
# Split the dataset into train and validation sets.
X = train_clean.drop(columns=TARGET_COLUMN)
y = train_clean[TARGET_COLUMN]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

numeric_transformer = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median"))]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=10)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)


def evaluate_model(model_name, model):
    model.fit(X_train, y_train)
    predictions = model.predict(X_val)
    probabilities = model.predict_proba(X_val)[:, 1]

    print(f"{model_name} Results")
    print(f"Accuracy:           {accuracy_score(y_val, predictions):.4f}")
    print(f"Balanced Accuracy:  {balanced_accuracy_score(y_val, predictions):.4f}")
    print(f"Precision:          {precision_score(y_val, predictions):.4f}")
    print(f"Recall:             {recall_score(y_val, predictions):.4f}")
    print(f"F1 Score:           {f1_score(y_val, predictions):.4f}")
    print(f"ROC AUC:            {roc_auc_score(y_val, probabilities):.4f}")
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_val, predictions))
    print("\nClassification Report:")
    print(classification_report(y_val, predictions))
    print()

    return {
        "model": model_name,
        "accuracy": accuracy_score(y_val, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_val, predictions),
        "precision": precision_score(y_val, predictions),
        "recall": recall_score(y_val, predictions),
        "f1": f1_score(y_val, predictions),
        "roc_auc": roc_auc_score(y_val, probabilities),
    }


## Decision Tree


In [27]:
decision_tree_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            DecisionTreeClassifier(
                max_depth=12,
                min_samples_leaf=25,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

decision_tree_results = evaluate_model("Decision Tree", decision_tree_model)


Decision Tree Results
Accuracy:           0.7995
Balanced Accuracy:  0.7898
Precision:          0.2098
Recall:             0.7787
F1 Score:           0.3305
ROC AUC:            0.8636

Confusion Matrix:
[[46076 11455]
 [  864  3041]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.80      0.88     57531
           1       0.21      0.78      0.33      3905

    accuracy                           0.80     61436
   macro avg       0.60      0.79      0.61     61436
weighted avg       0.93      0.80      0.85     61436




## Random Forest


In [28]:
random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=18,
                min_samples_leaf=8,
                class_weight="balanced_subsample",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

random_forest_results = evaluate_model("Random Forest", random_forest_model)


Random Forest Results
Accuracy:           0.7981
Balanced Accuracy:  0.7925
Precision:          0.2097
Recall:             0.7862
F1 Score:           0.3311
ROC AUC:            0.8767

Confusion Matrix:
[[45959 11572]
 [  835  3070]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.80      0.88     57531
           1       0.21      0.79      0.33      3905

    accuracy                           0.80     61436
   macro avg       0.60      0.79      0.61     61436
weighted avg       0.93      0.80      0.85     61436




## Boosted Tree


In [29]:
boosted_tree_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            AdaBoostClassifier(
                estimator=DecisionTreeClassifier(
                    max_depth=2,
                    min_samples_leaf=20,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                ),
                n_estimators=250,
                learning_rate=0.1,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

boosted_tree_results = evaluate_model("Boosted Tree", boosted_tree_model)


Boosted Tree Results
Accuracy:           0.8059
Balanced Accuracy:  0.7953
Precision:          0.2164
Recall:             0.7831
F1 Score:           0.3390
ROC AUC:            0.8778

Confusion Matrix:
[[46455 11076]
 [  847  3058]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.81      0.89     57531
           1       0.22      0.78      0.34      3905

    accuracy                           0.81     61436
   macro avg       0.60      0.80      0.61     61436
weighted avg       0.93      0.81      0.85     61436




## Model Comparison


In [30]:
results_df = pd.DataFrame([
    decision_tree_results,
    random_forest_results,
    boosted_tree_results,
]).sort_values("balanced_accuracy", ascending=False)

display(results_df)


,model,accuracy,balanced_accuracy,precision,recall,f1,roc_auc
2,Boosted Tree,0.805928,0.795288,0.216358,0.783099,0.339043,0.877822
1,Random Forest,0.798050,0.792514,0.209671,0.786172,0.331051,0.876665
0,Decision Tree,0.799482,0.789818,0.209782,0.778745,0.330526,0.863562
